In [ ]:
!pip install mlflow boto3 awscli

In [ ]:
!pip install optuna xgboost imbalanced-learn

In [ ]:
!aws configure

AWS Access Key ID [****************WTOX]: 
AWS Secret Access Key [****************+Fje]: 
Default region name [ap-south-1]: 
Default output format [None]: 


In [ ]:
import mlflow
mlflow.set_tracking_uri("http://ec2-13-201-115-239.ap-south-1.compute.amazonaws.com:5000/")
mlflow.set_experiment("Exp 5 -Trying out xgboost with hpt")

<Experiment: artifact_location='s3://youtube-sentiment-analysis-ahsulem-bucket/5', creation_time=1785412178276, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1785412178276, lifecycle_stage='active', name='Exp 5 -Trying out xgboost with hpt', tags={}, trace_location=None, workspace='default'>

In [ ]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import  LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv('/content/reddit_preprocessing.csv')
df.shape

(36793, 2)

In [ ]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])
df['clean_comment'] = df['clean_comment'].fillna('')
ngram_range = (1, 3)  # Trigram setting
max_features = 10000  # Set max_features to 1000 for TF-IDF

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

# Step 2: Vectorization using TF-IDF, fit on training data only
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
X_test_vec = vectorizer.transform(X_test)  # Transform test data

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))


# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)

# Run the experiment for XGBoost
run_optuna_experiment()


[I 2026-07-30 12:43:25,273] A new study created in memory with name: no-name-5070e1a1-dde8-4ffe-ac2f-f4c587dd45d3
[I 2026-07-30 12:44:24,646] Trial 0 finished with value: 0.7822736030828517 and parameters: {'n_estimators': 178, 'learning_rate': 0.012023707414106155, 'max_depth': 6}. Best is trial 0 with value: 0.7822736030828517.
[I 2026-07-30 12:46:41,812] Trial 1 finished with value: 0.7752671220879314 and parameters: {'n_estimators': 189, 'learning_rate': 0.0037601924322628786, 'max_depth': 10}. Best is trial 0 with value: 0.7822736030828517.
[I 2026-07-30 12:48:13,065] Trial 2 finished with value: 0.7349798563671396 and parameters: {'n_estimators': 141, 'learning_rate': 0.001266163752954646, 'max_depth': 10}. Best is trial 0 with value: 0.7822736030828517.
[I 2026-07-30 12:49:58,349] Trial 3 finished with value: 0.8397267472411981 and parameters: {'n_estimators': 210, 'learning_rate': 0.02059770943819904, 'max_depth': 10}. Best is trial 3 with value: 0.8397267472411981.
[I 2026-07-

🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://ec2-52-66-168-75.ap-south-1.compute.amazonaws.com:5000/#/experiments/5/runs/7b61f9233f7f4e79958f77b45b30f674
🧪 View experiment at: http://ec2-52-66-168-75.ap-south-1.compute.amazonaws.com:5000/#/experiments/5


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier'].